In [0]:
DECLARE OR REPLACE VARIABLE catalog_use STRING;
DECLARE OR REPLACE VARIABLE schema_use STRING;
DECLARE OR REPLACE VARIABLE table_use STRING;

In [0]:
SET VARIABLE catalog_use = :catalog_use;
SET VARIABLE schema_use = :schema_use;
SET VARIABLE table_use = :table_use;

In [0]:
USE IDENTIFIER(catalog_use || '.' || schema_use);
SELECT current_catalog(), current_schema();

In [0]:
DESCRIBE EXTENDED IDENTIFIER(catalog_use || '.' || schema_use || '.' || table_use) AS JSON;

In [0]:
DECLARE OR REPLACE VARIABLE metadata_json_string STRING;

SET VARIABLE metadata_json_string = DESCRIBE EXTENDED (catalog_use || '.' || schema_use || '.' || table_use) AS JSON;

In [0]:
FROM _sqldf |>
SELECT parse_json(json_metadata)

In [0]:
WITH description AS (
  t(DESCRIBE EXTENDED IDENTIFIER(catalog_use || '.' || schema_use || '.' || table_use) AS JSON)
)
SELECT parse_json(description.col_name) AS json_result
FROM description

In [0]:
-- with metadata as (
--   parse_json(DESCRIBE TABLE IDENTIFIER(catalog_use || '.' || schema_use || '.' || table_use) AS JSON)
-- )
-- SELECT 
--   *
-- FROM 
--   metadata
-- ;

In [0]:
CREATE OR REPLACE TEMPORARY FUNCTION get_json_metadata(
  catalog_name STRING COMMENT 'The name of the catalog for the table to return the extended metadata.'
  ,schema_name STRING COMMENT 'The name of the schema in the catalog, for the table to return the extended metadata.'
  ,table_name STRING COMMENT 'The name of the table in the catalog.schema to return the extended metadata.'
)
RETURNS STRING 
COMMENT 'Returns the extended metadata for the specified table as a JSON string. Not avaialble for Streaming Tables or Materialized Views.'
LANGUAGE PYTHON 
AS $$
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
return spark.sql(f"DESCRIBE EXTENDED {catalog_name}.{schema_name}.{table_name} AS JSON").first()[0]
$$;

In [0]:
SELECT get_json_metadata(catalog_use, schema_use, table_use);